# 11.4 圖片描述怎麼擴展成問答？

# 第 11 章：對齊、問答與語言能力保留

前置：第10章介面與第7章SFT。能力實驗需先有「文字屬性→正確答案」基模，以及可分類的視覺encoder；用scripts/prepare_data.py和pretrain_encoders.py準備。下面隨機模型只驗證資料／梯度，所有品質留給讀者訓練後量測。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：轉接頭尺寸合了，還要學資料的意思**

先確認文字基模會用文字屬性回答，再學圖片接頭。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/alignment.svg")))

**先想一想：**同一紅圓圖，問顏色與問形狀答案應一樣嗎？

caption只有一個描述目標；VQA同一圖會根據問題選擇不同屬性。資料中保留同圖多問，測試新組合，確認問題也有作用。

**把直覺寫成數學：**答案=f(image,question)，不是隻依圖片label。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
records = [
    {"image": "red_circle", "question": "color?", "answer": "red"},
    {"image": "red_circle", "question": "shape?", "answer": "circle"},
]
print(records)
assert records[0]["answer"] != records[1]["answer"]

**如何讀結果：**這是訓練schema；真實圖在synthetic manifest裏記錄生成seed與屬性，不能只喂問題答案。

**只改一件事：**只替換問題，保持圖片不變。
